In [21]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer 

In [13]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "fake_job_postings.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "shivamb/real-or-fake-fake-jobposting-prediction",
  file_path,
)



Note: you may need to restart the kernel to use updated packages.


C:\Users\Riju\AppData\Local\Temp\ipykernel_6724\2780137391.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


## Load data
### 2. Combine the text columns into one column called "text"

In [14]:
df["text"] = (
    df["title"].fillna("") + " "
    + df["company_profile"].fillna("") + " "
    + df["description"].fillna("") + " "
    + df["requirements"].fillna("")
)

df.head()

df.drop(columns = ['title', 'company_profile', 'description', 'requirements', 'job_id', 'location','department'
                   , 'benefits', 'salary_range', 'function', 'industry'], inplace = True)


C:\Users\Riju\AppData\Local\Temp\ipykernel_6724\534918432.py:10: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns = ['title', 'company_profile', 'description', 'requirements', 'job_id', 'location','department'


In [15]:
df.head(5)
df.isnull().sum()


telecommuting             0
has_company_logo          0
has_questions             0
employment_type        3471
required_experience    7050
required_education     8105
fraudulent                0
text                      0
dtype: int64

### 3. Fill missing values in categorical columns

In [16]:
cat_col = ['employment_type', 'required_experience', 'required_education']
for col in cat_col:
    df[col] = df[col].fillna('Unknown')

num_col = ['telecommuting', 'has_company_logo', 'has_questions']

### 4. Train / test split (stratify because fraud is rare)

In [17]:
X = df[['text'] + cat_col + num_col]
y = df['fraudulent']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify =y)

### 5. ColumnTransformer: different processing for different columns

In [18]:
preprocessor = ColumnTransformer(transformers = [
    ('text', CountVectorizer(), 'text'),
    ('cat', OneHotEncoder(handle_unknown ='ignore'), cat_col),
    ('num', 'passthrough', num_col)
])



### 6. Pipeline: preprocessing + model

In [22]:
model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter = 1000, class_weight = 'balanced'))
])
model.fit(x_train, y_train)


model_2 = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter = 1000, class_weight = 'balanced'))
])
model_2.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['text','employment_type','required_experience',...,'telecommuting', 'has_company_logo','has_questions']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By sp

### 7. Evaluate

In [20]:
y_pred = model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

joblib.dump(model, 'fake_job_post_detector_model.pkl')
print("Model saved as 'fake_job_post_detector_model.pkl'")

Accuracy: 0.9832214765100671
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3403
           1       0.81      0.86      0.83       173

    accuracy                           0.98      3576
   macro avg       0.90      0.93      0.91      3576
weighted avg       0.98      0.98      0.98      3576

Model saved as 'fake_job_post_detector_model.pkl'


### 8. Save